# 06 — Appendix: provenance of the 35-branch network

**Goal (PROBLEM.md, contribution 1):** the network is *rule-derived, not hand-picked*.
This notebook re-derives it from scratch out of the raw files and checks it matches
`config/nhs_network.json` exactly.

Selection rule audited:
1. **NHS trusts only** — 3-char `R`-prefixed org codes (excludes private/independent providers)
2. **Complete demand history** — Part_3 (new referrals) reported in all 60 months
3. **Region** = the branch's ICB at anchor month 2025-12, resolved live from the NHS ODS registry
4. **Top 5 per region** by mean monthly new referrals

Needs internet (ODS API) and ~5 minutes (scans ~5 GB of raw CSV).
Prior full runs 2026-07-23: **AUDIT PASSED** (172 trusts seen → 139 eligible → 35 selected, exact match).

In [1]:
import glob
import json
import os
import time
import pandas as pd
import requests

RAW_DIR = "../data/nhs_rtt_referral_to_treatment"
ANCHOR = "2025-12"  # fixed month for ICB membership (ICB boundaries changed over the years)
WINDOW = ("2021-04", "2026-03")  # selection window (frozen; extra files outside it are ignored)

USE = ["Provider Org Code", "Provider Parent Org Code", "RTT Part Type", "Treatment Function Code", "Total All"]

# 1. per-trust monthly demand totals across the selection window
files = [p for p in sorted(glob.glob(f"{RAW_DIR}/rtt_*.csv"))
         if WINDOW[0] <= os.path.basename(p)[4:11] <= WINDOW[1]]
print(f"scanning {len(files)} raw files")

frames = []
anchor_parent = None
for path in files:
    month = os.path.basename(path)[4:11]
    df = pd.read_csv(path, usecols=USE, dtype=str)
    df = df[df["Provider Org Code"].str.match(r"^R[A-Z0-9]{2}$")]      # NHS trusts only
    if month == ANCHOR:
        anchor_parent = df.drop_duplicates("Provider Org Code").set_index("Provider Org Code")["Provider Parent Org Code"]
    df = df[(df["RTT Part Type"] == "Part_3") & (df["Treatment Function Code"] != "C_999")]
    g = df.assign(v=pd.to_numeric(df["Total All"], errors="coerce").fillna(0)) \
          .groupby("Provider Org Code")["v"].sum().rename(month)
    frames.append(g)

panel = pd.concat(frames, axis=1)
print(f"NHS trusts ever seen: {len(panel)}")

scanning 60 raw files


NHS trusts ever seen: 172


In [2]:
# 2. completeness filter: demand reported in every window month
complete = panel.dropna()
complete = complete[(complete > 0).all(axis=1)]
print(f"trusts with demand in all {panel.shape[1]} months: {len(complete)}")

trusts with demand in all 60 months: 139


In [3]:
# 3. ICB -> NHS region, resolved live from the official NHS ODS registry (no hand-typed mapping)
icbs = sorted(set(anchor_parent.reindex(complete.index).dropna()))
print(f"distinct anchor ICBs: {len(icbs)}")

icb_region, region_name = {}, {}
for q in icbs:
    d = requests.get(f"https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/{q}", timeout=20).json()["Organisation"]
    region = [r["Target"]["OrgId"]["extension"] for r in d["Rels"]["Rel"]
              if r["Status"] == "Active" and r["id"] == "RE2"]
    icb_region[q] = region[0] if region else None
    time.sleep(0.15)
for y in sorted(set(icb_region.values())):
    d = requests.get(f"https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/{y}", timeout=20).json()["Organisation"]
    region_name[y] = d["Name"]
    time.sleep(0.15)
print(f"regions: {len(region_name)}")

distinct anchor ICBs: 42


regions: 7


In [4]:
# 4. rank by mean monthly demand, take top 5 per region, compare with config
res = pd.DataFrame({
    "mean_ref": complete.mean(axis=1).round(0),
    "icb": anchor_parent.reindex(complete.index),
})
res["region"] = res["icb"].map(icb_region).map(region_name)

with open("../config/nhs_network.json") as f:
    network = json.load(f)
cfg_sel = {t["code"] for r in network["regions"] for t in r["trusts"]}

derived_all = set()
for region, grp in res.groupby("region"):
    top5 = grp.sort_values("mean_ref", ascending=False).head(5)
    derived_all |= set(top5.index)
    print(f"\n{region} ({len(grp)} eligible trusts)")
    for code, row in top5.iterrows():
        flag = "OK" if code in cfg_sel else "NOT-IN-CONFIG"
        print(f"  {code}  mean_ref={int(row['mean_ref']):>6}  {flag}")

print(f"\nconfig trusts not reproduced by audit: {sorted(cfg_sel - derived_all) or 'none'}")
print(f"audit trusts not in config:            {sorted(derived_all - cfg_sel) or 'none'}")
print("AUDIT PASSED" if derived_all == cfg_sel else "AUDIT FOUND DIFFERENCES")


EAST OF ENGLAND COMMISSIONING REGION (15 eligible trusts)
  RAJ  mean_ref= 40077  OK
  RDE  mean_ref= 18562  OK
  RC9  mean_ref= 18528  OK
  RM1  mean_ref= 16045  OK
  RGT  mean_ref= 14599  OK

LONDON COMMISSIONING REGION (21 eligible trusts)
  RJ1  mean_ref= 27407  OK
  RJZ  mean_ref= 23590  OK
  R1H  mean_ref= 23475  OK
  RYJ  mean_ref= 22323  OK
  RAL  mean_ref= 21729  OK

MIDLANDS COMMISSIONING REGION (27 eligible trusts)
  RRK  mean_ref= 24823  OK
  RTG  mean_ref= 21751  OK
  RWE  mean_ref= 21190  OK
  RX1  mean_ref= 18097  OK
  RXK  mean_ref= 16144  OK

NORTH EAST AND YORKSHIRE COMMISSIONING REGION (21 eligible trusts)
  RTD  mean_ref= 28396  OK
  RR8  mean_ref= 23553  OK
  RWA  mean_ref= 18519  OK
  R0B  mean_ref= 18494  OK
  RXF  mean_ref= 14340  OK

NORTH WEST COMMISSIONING REGION (21 eligible trusts)
  RM3  mean_ref= 23203  OK
  RBN  mean_ref= 16663  OK
  REM  mean_ref= 15659  OK
  RXR  mean_ref= 13018  OK
  RXN  mean_ref= 11734  OK

SOUTH EAST COMMISSIONING REGION (20 eligi